# Building a Recurrent Neural Network From Scratch in Python

Runnable NumPy companion for the complete original forward/backward tutorial. Run all cells in a fresh Python3.11+ kernel; no dataset downloads, API keys, GPU or deep-learning framework are needed. The first five function examples follow the source, with the missing NumPy setup and deterministic executable demonstrations added. The one mathematical maintenance change is column-wise stable softmax, not a batch-global maximum. Original functions are preserved verbatim in the nonexecuting source appendix.

The original backward functions differentiate hidden states, not the softmax output projection. Independent finite-difference checks cover the entire hidden recurrence and the output-head extension. A separately labeled companion-only training example adds cross-entropy, output gradients and SGD on a synthetic delayed-bit task; it is not an original article experiment, real-world benchmark or reported model accuracy. No source training workload is shortened: the source defines functions but contains no dataset, training loop or epoch count.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
               "numpy==2.4.6", "matplotlib==3.10.9"], check=True)
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=6, suppress=True)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)

In [ ]:
rng = np.random.default_rng(42)
n_x, n_a, n_y, m, T_x = 3, 5, 2, 2, 4
x = rng.normal(size=(n_x, m, T_x))
a0 = rng.normal(size=(n_a, m))
parameters = {
    "Wax": rng.normal(scale=0.2, size=(n_a, n_x)),
    "Waa": rng.normal(scale=0.2, size=(n_a, n_a)),
    "Wya": rng.normal(scale=0.2, size=(n_y, n_a)),
    "ba": np.zeros((n_a, 1)),
    "by": np.zeros((n_y, 1)),
}
print("Input / initial hidden shapes:", x.shape, a0.shape)
print("Dimensions: n_x=3, n_a=5, n_y=2, batch=2, time steps=4")

## Stable softmax

In [ ]:
def softmax(x):
    e_x = np.exp(x - np.max(x, axis=0, keepdims=True))
    return e_x / e_x.sum(axis=0, keepdims=True)

scores = np.array([[1000.0, -1000.0], [999.0, -1001.0]])
print("Column-wise probabilities:")
print(softmax(scores))
print("Column sums:", softmax(scores).sum(axis=0))

## Single RNN cell forward pass

In [ ]:
def rnn_cell_forward(xt, a_prev, parameters):
    """
    Implements a single forward step of the RNN-cell as described in Figure (2)

    Arguments:
    xt -- your input data at timestep "t", numpy array of shape (n_x, m).
    a_prev -- Hidden state at timestep "t-1", numpy array of shape (n_a, m)
    parameters -- python dictionary containing:
                        Wax -- Weight matrix multiplying the input, numpy array of shape (n_a, n_x)
                        Waa -- Weight matrix multiplying the hidden state, numpy array of shape (n_a, n_a)
                        Wya -- Weight matrix relating the hidden-state to the output, numpy array of shape (n_y, n_a)
                        ba --  Bias, numpy array of shape (n_a, 1)
                        by -- Bias relating the hidden-state to the output, numpy array of shape (n_y, 1)
    Returns:
    a_next -- next hidden state, of shape (n_a, m)
    yt_pred -- prediction at timestep "t", numpy array of shape (n_y, m)
    cache -- tuple of values needed for the backward pass, contains (a_next, a_prev, xt, parameters)
    """
    
    # Retrieve parameters from "parameters"
    Wax = parameters["Wax"]
    Waa = parameters["Waa"]
    Wya = parameters["Wya"]
    ba = parameters["ba"]
    by = parameters["by"]
    
    # compute next activation state using the formula given above
    a_next = np.tanh(np.dot(Wax,xt) + np.dot(Waa,a_prev) + ba)
    # compute output of the current cell using the formula given above
    yt_pred = softmax(np.dot(Wya,a_next) + by) 
    
    # store values you need for backward propagation in cache
    cache = (a_next, a_prev, xt, parameters)
    
    return a_next, yt_pred, cache

a_next, yt_pred, cell_cache = rnn_cell_forward(x[:, :, 0], a0, parameters)
print("Hidden state:")
print(a_next)
print("Prediction:")
print(yt_pred)
print("Cache entries:", len(cell_cache))

## Complete sequence forward pass

In [ ]:
def rnn_forward(x, a0, parameters):
    """
    Implement the forward propagation of the recurrent neural network described in Figure (3).

    Arguments:
    x -- Input data for every time-step, of shape (n_x, m, T_x).
    a0 -- Initial hidden state, of shape (n_a, m)
    parameters -- python dictionary containing:
                        Waa -- Weight matrix multiplying the hidden state, numpy array of shape (n_a, n_a)
                        Wax -- Weight matrix multiplying the input, numpy array of shape (n_a, n_x)
                        Wya -- Weight matrix relating the hidden-state to the output, numpy array of shape (n_y, n_a)
                        ba --  Bias numpy array of shape (n_a, 1)
                        by -- Bias relating the hidden-state to the output, numpy array of shape (n_y, 1)

    Returns:
    a -- Hidden states for every time-step, numpy array of shape (n_a, m, T_x)
    y_pred -- Predictions for every time-step, numpy array of shape (n_y, m, T_x)
    caches -- tuple of values needed for the backward pass, contains (list of caches, x)
    """
    
    # Initialize "caches" which will contain the list of all caches
    caches = []
    
    # Retrieve dimensions from shapes of x and parameters["Wya"]
    n_x, m, T_x = x.shape
    n_y, n_a = parameters["Wya"].shape
       
    # initialize "a" and "y_pred" with zeros (≈2 lines)
    a = np.zeros((n_a, m, T_x))
    y_pred = np.zeros((n_y, m, T_x))
    
    # Initialize a_next (≈1 line)
    a_next = a0
    
    # loop over all time-steps
    for t in range(T_x):
        # Update next hidden state, compute the prediction, get the cache (≈1 line)
        a_next, yt_pred, cache = rnn_cell_forward(x[:,:,t], a_next, parameters)
        # Save the value of the new "next" hidden state in a (≈1 line)
        a[:,:,t] = a_next
        # Save the value of the prediction in y (≈1 line)
        y_pred[:,:,t] = yt_pred
        # Append "cache" to "caches" (≈1 line)
        caches.append(cache)
    
    # store values needed for backward propagation in cache
    caches = (caches, x)
    
    return a, y_pred, caches

a, y_pred, sequence_cache = rnn_forward(x, a0, parameters)
print("Hidden / prediction shapes:", a.shape, y_pred.shape)
print("Final-step probabilities:")
print(y_pred[:, :, -1])
print("Cached time steps:", len(sequence_cache[0]))

## Single RNN cell backward pass

In [ ]:
def rnn_cell_backward(da_next, cache):
    """
    Implements the backward pass for the RNN-cell (single time-step).

    Arguments:
    da_next -- Gradient of loss with respect to next hidden state
    cache -- python dictionary containing useful values (output of rnn_cell_forward())

    Returns:
    gradients -- python dictionary containing:
                        dx -- Gradients of input data, of shape (n_x, m)
                        da_prev -- Gradients of previous hidden state, of shape (n_a, m)
                        dWax -- Gradients of input-to-hidden weights, of shape (n_a, n_x)
                        dWaa -- Gradients of hidden-to-hidden weights, of shape (n_a, n_a)
                        dba -- Gradients of bias vector, of shape (n_a, 1)
    """
    
    # Retrieve values from cache
    (a_next, a_prev, xt, parameters) = cache
    
    # Retrieve values from parameters
    Wax = parameters["Wax"]
    Waa = parameters["Waa"]
    Wya = parameters["Wya"]
    ba = parameters["ba"]
    by = parameters["by"]

    # compute the gradient of tanh with respect to a_next (≈1 line)
    dtanh = (1-a_next*a_next)*da_next

    # compute the gradient of the loss with respect to Wax (≈2 lines)
    dxt = np.dot(Wax.T,  dtanh)
    dWax = np.dot(dtanh,xt.T)

    # compute the gradient with respect to Waa (≈2 lines)
    da_prev = np.dot(Waa.T, dtanh)  
    dWaa = np.dot( dtanh,a_prev.T)

    # compute the gradient with respect to b (≈1 line)
    dba = np.sum( dtanh,keepdims=True,axis=-1)

    
    # Store the gradients in a python dictionary
    gradients = {"dxt": dxt, "da_prev": da_prev, "dWax": dWax, "dWaa": dWaa, "dba": dba}
    

    return gradients

da_next = rng.normal(size=a_next.shape)
cell_gradients = rnn_cell_backward(da_next, cell_cache)
for name, value in cell_gradients.items():
    print(name, "shape:", value.shape, "| norm:", f"{np.linalg.norm(value):.6f}")

## Backpropagation through time

In [ ]:
def rnn_backward(da, caches):
    """
    Implement the backward pass for a RNN over an entire sequence of input data.

    Arguments:
    da -- Upstream gradients of all hidden states, of shape (n_a, m, T_x)
    caches -- tuple containing information from the forward pass (rnn_forward)
    
    Returns:
    gradients -- python dictionary containing:
                        dx -- Gradient w.r.t. the input data, numpy-array of shape (n_x, m, T_x)
                        da0 -- Gradient w.r.t the initial hidden state, numpy-array of shape (n_a, m)
                        dWax -- Gradient w.r.t the input's weight matrix, numpy-array of shape (n_a, n_x)
                        dWaa -- Gradient w.r.t the hidden state's weight matrix, numpy array of shape (n_a, n_a)
                        dba -- Gradient w.r.t the bias, of shape (n_a, 1)
    """
        
    ### START CODE HERE ###
    
    # Retrieve values from the first cache (t=1) of caches (≈2 lines)
    (caches, x) = caches
    (a1, a0, x1, parameters) = caches[0]
    
    # Retrieve dimensions from da's and x1's shapes (≈2 lines)
    n_a, m, T_x = da.shape
    n_x, m = x1.shape 
    
    # initialize the gradients with the right sizes (≈6 lines)
    dx = np.zeros((n_x, m, T_x)) 
    dWax = np.zeros((n_a, n_x))
    dWaa = np.zeros((n_a, n_a))
    dba = np.zeros((n_a, 1)) 
    da0 = np.zeros((n_a, m))
    da_prevt = np.zeros((n_a, m))  
    
    # Loop through all the time steps
    for t in reversed(range(T_x)):
        # Compute gradients at time step t. Choose wisely the "da_next" and the "cache" to use in the backward propagation step. (≈1 line)
        gradients = rnn_cell_backward(da[:, :, t] + da_prevt, caches[t])
        # Retrieve derivatives from gradients (≈ 1 line)
        dxt, da_prevt, dWaxt, dWaat, dbat = gradients["dxt"], gradients["da_prev"], gradients["dWax"], gradients["dWaa"], gradients["dba"]
        # Increment global derivatives w.r.t parameters by adding their derivative at time-step t (≈4 lines)
        dx[:, :, t] = dxt  
        dWax += dWaxt  
        dWaa += dWaat  
        dba += dbat  
        
    # Set da0 to the gradient of a which has been backpropagated through all time-steps (≈1 line) 
    da0 = da_prevt
    ### END CODE HERE ###

    # Store the gradients in a python dictionary
    gradients = {"dx": dx, "da0": da0, "dWax": dWax, "dWaa": dWaa,"dba": dba}
    
    return gradients

da = rng.normal(size=a.shape)
sequence_gradients = rnn_backward(da, sequence_cache)
for name, value in sequence_gradients.items():
    print(name, "shape:", value.shape, "| norm:", f"{np.linalg.norm(value):.6f}")

## Independent gradient and recurrence checks

Central finite differences check every scalar component, including input/initial hidden state and shared recurrent parameters, for three seeds. Cell objectives are sum(a_next*upstream), sequence objectives sum(a*upstream). A separate scalar-loop forward implementation checks the batched implementation. Check softmax on widely separated column ranges, one-step recurrence, batch permutation, untouched inputs and shared parameter identities. Finally verify cross-entropy/output gradients for the separately labeled training extension.

In [ ]:
def numerical_gradient(value, objective, epsilon=1e-6):
    result = np.zeros_like(value)
    for index in np.ndindex(value.shape):
        saved = value[index]
        try:
            value[index] = saved + epsilon
            plus = objective()
            value[index] = saved - epsilon
            minus = objective()
        finally:
            value[index] = saved
        result[index] = (plus - minus) / (2 * epsilon)
    return result

def scalar_forward(inputs, initial, params):
    nx, batch, steps = inputs.shape
    ny, na = params["Wya"].shape
    states = np.zeros((na, batch, steps))
    predictions = np.zeros((ny, batch, steps))
    for sample in range(batch):
        previous = initial[:, sample].copy()
        for t in range(steps):
            current = np.array([np.tanh(sum(params["Wax"][i, j] * inputs[j, sample, t] for j in range(nx))
                                + sum(params["Waa"][i, j] * previous[j] for j in range(na))
                                + params["ba"][i, 0]) for i in range(na)])
            logits = np.array([sum(params["Wya"][i, j] * current[j] for j in range(na))
                               + params["by"][i, 0] for i in range(ny)])
            exps = np.exp(logits - max(logits))
            states[:, sample, t] = current
            predictions[:, sample, t] = exps / sum(exps)
            previous = current
    return states, predictions

def classification_gradients(inputs, initial, params, labels):
    states, probabilities, cache = rnn_forward(inputs, initial, params)
    logits = np.einsum("ij,jmt->imt", params["Wya"], states) + params["by"][:, :, None]
    shifted = logits - logits.max(axis=0, keepdims=True)
    log_probs = shifted - np.log(np.exp(shifted).sum(axis=0, keepdims=True))
    count = labels.shape[0] * labels.shape[1]
    targets = np.eye(probabilities.shape[0])[labels].transpose(2, 0, 1)
    loss = float(-np.sum(targets * log_probs) / count)
    dlogits = (probabilities - targets) / count
    upstream = np.einsum("ji,jmt->imt", params["Wya"], dlogits)
    gradients = rnn_backward(upstream, cache)
    gradients["dWya"] = np.einsum("imt,jmt->ij", dlogits, states)
    gradients["dby"] = dlogits.sum(axis=(1, 2))[:, None]
    return loss, gradients

max_error = 0.0
checked_components = 0
for seed in [0, 1, 42]:
    random = np.random.default_rng(seed)
    inputs = random.normal(scale=0.3, size=(2, 2, 3))
    initial = random.normal(scale=0.3, size=(3, 2))
    params = {"Wax": random.normal(scale=0.2, size=(3, 2)),
              "Waa": random.normal(scale=0.2, size=(3, 3)),
              "Wya": random.normal(scale=0.2, size=(2, 3)),
              "ba": random.normal(scale=0.2, size=(3, 1)),
              "by": random.normal(scale=0.2, size=(2, 1))}
    originals = [v.copy() for v in [inputs, initial, *params.values()]]
    states, probabilities, cache = rnn_forward(inputs, initial, params)
    scalar_states, scalar_probabilities = scalar_forward(inputs, initial, params)
    np.testing.assert_allclose(states, scalar_states, atol=1e-14)
    np.testing.assert_allclose(probabilities, scalar_probabilities, atol=1e-14)
    np.testing.assert_allclose(probabilities.sum(axis=0), 1, atol=1e-14)
    assert len(cache[0]) == 3 and all(c[3] is params for c in cache[0])
    single, single_probs, _ = rnn_cell_forward(inputs[:, :, 0], initial, params)
    one, one_probs, _ = rnn_forward(inputs[:, :, :1], initial, params)
    np.testing.assert_array_equal(one[:, :, 0], single)
    np.testing.assert_array_equal(one_probs[:, :, 0], single_probs)
    permuted, permuted_probs, _ = rnn_forward(inputs[:, ::-1, :], initial[:, ::-1], params)
    np.testing.assert_allclose(permuted, states[:, ::-1, :], atol=1e-14)
    np.testing.assert_allclose(permuted_probs, probabilities[:, ::-1, :], atol=1e-14)
    for v, saved in zip([inputs, initial, *params.values()], originals):np.testing.assert_array_equal(v, saved)

    cell_upstream = random.normal(size=single.shape)
    cell_grads = rnn_cell_backward(cell_upstream, cache[0][0])
    cell_input = inputs[:, :, 0]
    cell_objective = lambda: float(np.sum(rnn_cell_forward(cell_input, initial, params)[0] * cell_upstream))
    upstream = random.normal(size=states.shape)
    seq_grads = rnn_backward(upstream, cache)
    seq_objective = lambda: float(np.sum(rnn_forward(inputs, initial, params)[0] * upstream))
    for values, gradients, objective in [
        ({"dxt": cell_input, "da_prev": initial, "dWax": params["Wax"], "dWaa": params["Waa"], "dba": params["ba"]}, cell_grads, cell_objective),
        ({"dx": inputs, "da0": initial, "dWax": params["Wax"], "dWaa": params["Waa"], "dba": params["ba"]}, seq_grads, seq_objective),
    ]:
        for name, value in values.items():
            numerical = numerical_gradient(value, objective)
            max_error = max(max_error, float(np.max(np.abs(numerical - gradients[name]))))
            np.testing.assert_allclose(gradients[name], numerical, atol=2e-8, rtol=2e-6)
            checked_components += value.size

    labels = random.integers(0, 2, size=(2, 3))
    loss, full_grads = classification_gradients(inputs, initial, params, labels)
    full_objective = lambda: classification_gradients(inputs, initial, params, labels)[0]
    for name, value in {"dx": inputs, "da0": initial, **{"d" + k: v for k, v in params.items()}}.items():
        numerical = numerical_gradient(value, full_objective)
        max_error = max(max_error, float(np.max(np.abs(numerical - full_grads[name]))))
        np.testing.assert_allclose(full_grads[name], numerical, atol=2e-8, rtol=2e-6)
        checked_components += value.size
    print("Seed", seed, ": scalar forward, shared cache, single step, batch permutation and all gradients passed.")

extreme = np.array([[1000.0, -1000.0], [999.0, -1001.0]])
expected = np.array([[1 / (1 + np.exp(-1))] * 2, [1 / (1 + np.exp(1))] * 2])
np.testing.assert_allclose(softmax(extreme), expected, atol=1e-14)
np.testing.assert_allclose(softmax(extreme + np.array([[5000.0, -8000.0]])), expected, atol=1e-14)
print("Central-difference scalar components:", checked_components)
print("Maximum absolute gradient error:", f"{max_error:.3e}")
print("All independent RNN forward/backward and output-head regression checks passed.")

## Companion-only extension: train a delayed-bit classifier

This is additional executable integration coverage, not a source experiment. Each target is the previous input bit, with zero as the first target. Train a six-hidden-unit tanh RNN for 300 full-batch SGD updates on 64 generated sequences of length6, then evaluate on 128 separately generated sequences. Cross-entropy/output gradients above complete the trainable head while reusing the original RNN recurrence and BPTT. No real dataset/benchmark claim is made.

In [ ]:
training_rng = np.random.default_rng(7)
def delayed_bits(count):
    bits = training_rng.integers(0, 2, size=(count, 6))
    inputs = np.eye(2)[bits].transpose(2, 0, 1)
    labels = np.concatenate([np.zeros((count, 1), dtype=int), bits[:, :-1]], axis=1)
    return inputs, labels

train_x, train_y = delayed_bits(64)
test_x, test_y = delayed_bits(128)
train_a0 = np.zeros((6, 64))
test_a0 = np.zeros((6, 128))
train_params = {"Wax": training_rng.normal(scale=0.2, size=(6, 2)),
                "Waa": training_rng.normal(scale=0.2, size=(6, 6)),
                "Wya": training_rng.normal(scale=0.2, size=(2, 6)),
                "ba": np.zeros((6, 1)), "by": np.zeros((2, 1))}
initial_loss = classification_gradients(train_x, train_a0, train_params, train_y)[0]
losses = []
for step in range(300):
    loss, gradients = classification_gradients(train_x, train_a0, train_params, train_y)
    assert np.isfinite(loss) and all(np.isfinite(v).all() for v in gradients.values())
    losses.append(loss)
    for key in train_params:train_params[key] -= 0.5 * gradients["d" + key]
final_loss = classification_gradients(train_x, train_a0, train_params, train_y)[0]
test_loss = classification_gradients(test_x, test_a0, train_params, test_y)[0]
test_predictions = rnn_forward(test_x, test_a0, train_params)[1].argmax(axis=0)
accuracy = float(np.mean(test_predictions == test_y))
assert final_loss < initial_loss * 0.1 and accuracy > 0.95
print("Synthetic delayed-bit task | training sequences: 64 | test sequences: 128 | time steps: 6")
print("Full-batch SGD updates: 300 | learning rate: 0.5")
print(f"Training loss: {initial_loss:.6f} -> {final_loss:.6f}")
print(f"Separate synthetic test loss: {test_loss:.6f} | token accuracy: {accuracy:.6f}")
plt.figure(figsize=(8, 4))
plt.plot(losses, color="#174dba")
plt.xlabel("Full-batch SGD update")
plt.ylabel("Mean cross-entropy")
plt.title("Companion-only synthetic delayed-bit training")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Immutable original inline code

Recovered from the source HTML at SHA-256 6d4d3d71f9f420136374caa002363350b393f1350ac0b1a148f1fb963675b21d. Original HTML line breaks restored without changing tokens. These historical Markdown blocks are not executed.

### Source code 1

```python
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)
```

### Source code 2

```python
def rnn_cell_forward(xt, a_prev, parameters):
    """
    Implements a single forward step of the RNN-cell as described in Figure (2)

    Arguments:
    xt -- your input data at timestep "t", numpy array of shape (n_x, m).
    a_prev -- Hidden state at timestep "t-1", numpy array of shape (n_a, m)
    parameters -- python dictionary containing:
                        Wax -- Weight matrix multiplying the input, numpy array of shape (n_a, n_x)
                        Waa -- Weight matrix multiplying the hidden state, numpy array of shape (n_a, n_a)
                        Wya -- Weight matrix relating the hidden-state to the output, numpy array of shape (n_y, n_a)
                        ba --  Bias, numpy array of shape (n_a, 1)
                        by -- Bias relating the hidden-state to the output, numpy array of shape (n_y, 1)
    Returns:
    a_next -- next hidden state, of shape (n_a, m)
    yt_pred -- prediction at timestep "t", numpy array of shape (n_y, m)
    cache -- tuple of values needed for the backward pass, contains (a_next, a_prev, xt, parameters)
    """
    
    # Retrieve parameters from "parameters"
    Wax = parameters["Wax"]
    Waa = parameters["Waa"]
    Wya = parameters["Wya"]
    ba = parameters["ba"]
    by = parameters["by"]
    
    # compute next activation state using the formula given above
    a_next = np.tanh(np.dot(Wax,xt) + np.dot(Waa,a_prev) + ba)
    # compute output of the current cell using the formula given above
    yt_pred = softmax(np.dot(Wya,a_next) + by) 
    
    # store values you need for backward propagation in cache
    cache = (a_next, a_prev, xt, parameters)
    
    return a_next, yt_pred, cache
```

### Source code 3

```python
def rnn_forward(x, a0, parameters):
    """
    Implement the forward propagation of the recurrent neural network described in Figure (3).

    Arguments:
    x -- Input data for every time-step, of shape (n_x, m, T_x).
    a0 -- Initial hidden state, of shape (n_a, m)
    parameters -- python dictionary containing:
                        Waa -- Weight matrix multiplying the hidden state, numpy array of shape (n_a, n_a)
                        Wax -- Weight matrix multiplying the input, numpy array of shape (n_a, n_x)
                        Wya -- Weight matrix relating the hidden-state to the output, numpy array of shape (n_y, n_a)
                        ba --  Bias numpy array of shape (n_a, 1)
                        by -- Bias relating the hidden-state to the output, numpy array of shape (n_y, 1)

    Returns:
    a -- Hidden states for every time-step, numpy array of shape (n_a, m, T_x)
    y_pred -- Predictions for every time-step, numpy array of shape (n_y, m, T_x)
    caches -- tuple of values needed for the backward pass, contains (list of caches, x)
    """
    
    # Initialize "caches" which will contain the list of all caches
    caches = []
    
    # Retrieve dimensions from shapes of x and parameters["Wya"]
    n_x, m, T_x = x.shape
    n_y, n_a = parameters["Wya"].shape
       
    # initialize "a" and "y_pred" with zeros (≈2 lines)
    a = np.zeros((n_a, m, T_x))
    y_pred = np.zeros((n_y, m, T_x))
    
    # Initialize a_next (≈1 line)
    a_next = a0
    
    # loop over all time-steps
    for t in range(T_x):
        # Update next hidden state, compute the prediction, get the cache (≈1 line)
        a_next, yt_pred, cache = rnn_cell_forward(x[:,:,t], a_next, parameters)
        # Save the value of the new "next" hidden state in a (≈1 line)
        a[:,:,t] = a_next
        # Save the value of the prediction in y (≈1 line)
        y_pred[:,:,t] = yt_pred
        # Append "cache" to "caches" (≈1 line)
        caches.append(cache)
    
    # store values needed for backward propagation in cache
    caches = (caches, x)
    
    return a, y_pred, caches
```

### Source code 4

```python
def rnn_cell_backward(da_next, cache):
    """
    Implements the backward pass for the RNN-cell (single time-step).

    Arguments:
    da_next -- Gradient of loss with respect to next hidden state
    cache -- python dictionary containing useful values (output of rnn_cell_forward())

    Returns:
    gradients -- python dictionary containing:
                        dx -- Gradients of input data, of shape (n_x, m)
                        da_prev -- Gradients of previous hidden state, of shape (n_a, m)
                        dWax -- Gradients of input-to-hidden weights, of shape (n_a, n_x)
                        dWaa -- Gradients of hidden-to-hidden weights, of shape (n_a, n_a)
                        dba -- Gradients of bias vector, of shape (n_a, 1)
    """
    
    # Retrieve values from cache
    (a_next, a_prev, xt, parameters) = cache
    
    # Retrieve values from parameters
    Wax = parameters["Wax"]
    Waa = parameters["Waa"]
    Wya = parameters["Wya"]
    ba = parameters["ba"]
    by = parameters["by"]

    # compute the gradient of tanh with respect to a_next (≈1 line)
    dtanh = (1-a_next*a_next)*da_next

    # compute the gradient of the loss with respect to Wax (≈2 lines)
    dxt = np.dot(Wax.T,  dtanh)
    dWax = np.dot(dtanh,xt.T)

    # compute the gradient with respect to Waa (≈2 lines)
    da_prev = np.dot(Waa.T, dtanh)  
    dWaa = np.dot( dtanh,a_prev.T)

    # compute the gradient with respect to b (≈1 line)
    dba = np.sum( dtanh,keepdims=True,axis=-1)

    
    # Store the gradients in a python dictionary
    gradients = {"dxt": dxt, "da_prev": da_prev, "dWax": dWax, "dWaa": dWaa, "dba": dba}
    

    return gradients
```

### Source code 5

```python
def rnn_backward(da, caches):
    """
    Implement the backward pass for a RNN over an entire sequence of input data.

    Arguments:
    da -- Upstream gradients of all hidden states, of shape (n_a, m, T_x)
    caches -- tuple containing information from the forward pass (rnn_forward)
    
    Returns:
    gradients -- python dictionary containing:
                        dx -- Gradient w.r.t. the input data, numpy-array of shape (n_x, m, T_x)
                        da0 -- Gradient w.r.t the initial hidden state, numpy-array of shape (n_a, m)
                        dWax -- Gradient w.r.t the input's weight matrix, numpy-array of shape (n_a, n_x)
                        dWaa -- Gradient w.r.t the hidden state's weight matrix, numpy array of shape (n_a, n_a)
                        dba -- Gradient w.r.t the bias, of shape (n_a, 1)
    """
        
    ### START CODE HERE ###
    
    # Retrieve values from the first cache (t=1) of caches (≈2 lines)
    (caches, x) = caches
    (a1, a0, x1, parameters) = caches[0]
    
    # Retrieve dimensions from da's and x1's shapes (≈2 lines)
    n_a, m, T_x = da.shape
    n_x, m = x1.shape 
    
    # initialize the gradients with the right sizes (≈6 lines)
    dx = np.zeros((n_x, m, T_x)) 
    dWax = np.zeros((n_a, n_x))
    dWaa = np.zeros((n_a, n_a))
    dba = np.zeros((n_a, 1)) 
    da0 = np.zeros((n_a, m))
    da_prevt = np.zeros((n_a, m))  
    
    # Loop through all the time steps
    for t in reversed(range(T_x)):
        # Compute gradients at time step t. Choose wisely the "da_next" and the "cache" to use in the backward propagation step. (≈1 line)
        gradients = rnn_cell_backward(da[:, :, t] + da_prevt, caches[t])
        # Retrieve derivatives from gradients (≈ 1 line)
        dxt, da_prevt, dWaxt, dWaat, dbat = gradients["dxt"], gradients["da_prev"], gradients["dWax"], gradients["dWaa"], gradients["dba"]
        # Increment global derivatives w.r.t parameters by adding their derivative at time-step t (≈4 lines)
        dx[:, :, t] = dxt  
        dWax += dWaxt  
        dWaa += dWaat  
        dba += dbat  
        
    # Set da0 to the gradient of a which has been backpropagated through all time-steps (≈1 line) 
    da0 = da_prevt
    ### END CODE HERE ###

    # Store the gradients in a python dictionary
    gradients = {"dx": dx, "da0": da0, "dWax": dWax, "dWaa": dWaa,"dba": dba}
    
    return gradients
```